In [ ]:
!pip install -q -U sentence-transformers


In [ ]:
import glob, os, shutil, zipfile

def find_source(root='/kaggle/input', max_depth=4):
    found, archives = [], []
    for cur, dirs, files in os.walk(root):
        if cur.count('/') - root.count('/') >= max_depth:
            dirs.clear()
        if 'rufin' in dirs:
            stale = ('__notebook__.ipynb' in files or 'embeddings' in dirs
                     or '__output__.json' in files)
            found.append((1 if stale else 0, cur))
        archives += [os.path.join(cur, f) for f in files if f.endswith('.zip')]
    if found:
        found.sort()
        return found[0][1], None
    return (os.path.dirname(archives[0]), archives[0]) if archives else (None, None)

src, archive = find_source()
assert src, 'датасет ru-fin-retrieval не подключён'
print('источник:', src)
if archive:
    with zipfile.ZipFile(archive) as z:
        z.extractall('/kaggle/working')
else:
    for item in glob.glob(os.path.join(src, '*')):
        dst = os.path.join('/kaggle/working', os.path.basename(item))
        if os.path.isdir(item):
            shutil.copytree(item, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(item, dst)
%cd /kaggle/working
print('в рабочей папке:', sorted(os.listdir('.'))[:12])


In [ ]:
!cd /kaggle/working && python export_chunks.py
